# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [ ]:
dataset_path = 'auto_dataset.csv'

if os.path.exists(dataset_path):
    data = pd.read_csv(dataset_path)
else:
    # Генерация аналогичного датасета, если файла нет
    rng_data = np.random.default_rng(GLOBAL_SEED)
    n = 1000
    data = pd.DataFrame({
        'brand': rng_data.choice(['bmw', 'audi', 'vw', 'mercedes'], n),
        'model': rng_data.choice(['x5', 'a4', 'golf', 'c_class'], n),
        'vehicleType': rng_data.choice(['sedan', 'suv', 'coupe'], n),
        'gearbox': rng_data.choice(['manual', 'automatic'], n),
        'fuelType': rng_data.choice(['petrol', 'diesel'], n),
        'notRepairedDamage': rng_data.choice(['yes', 'no'], n),
        'powerPS': rng_data.integers(60, 400, n),
        'kilometer': rng_data.choice([15000, 50000, 100000, 150000], n),
        'autoAgeMonths': rng_data.integers(6, 240, n),
        'price': rng_data.integers(1000, 50000, n)
    })

data = data.dropna()
data = data[data['price'] > 0]

print("--- 1. Первые строки датасета ---")
print(data.head())

2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [ ]:
# 2. One-Hot Encoding категориальных признаков
full_encoded = pd.get_dummies(
    pd.concat([train_data, val_data, test_data]),
    columns=['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage'],
    drop_first=True
)

train_enc = full_encoded.iloc[:len(train_data)]
val_enc = full_encoded.iloc[len(train_data):len(train_data) + len(val_data)]
test_enc = full_encoded.iloc[len(train_data) + len(val_data):]

X_tr_raw = train_enc.drop(columns=[target_col]).values.astype(float)
y_tr_raw = train_enc[target_col].values.reshape(-1, 1).astype(float)

X_v_raw = val_enc.drop(columns=[target_col]).values.astype(float)
y_v_raw = val_enc[target_col].values.reshape(-1, 1).astype(float)

X_te_raw = test_enc.drop(columns=[target_col]).values.astype(float)
y_te_raw = test_enc[target_col].values.reshape(-1, 1).astype(float)

# Масштабирование признаков (X) с помощью StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_tr_raw)
X_val = scaler_X.transform(X_v_raw)
X_test = scaler_X.transform(X_te_raw)

# Ответы (y) оставляем в исходной шкале
y_train, y_val, y_test = y_tr_raw, y_v_raw, y_te_raw

# Добавление единичного столбца для bias (свободного коэффициента)
X_train = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
X_val = np.hstack([np.ones((X_val.shape[0], 1)), X_val])
X_test = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

print(f"Размеры выборок: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")

3. Разбейте датасет на train val test в отношении 8:1:1

In [ ]:
target_col = 'price'

# 3. Разбиение на train, val, test (8:1:1)
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=GLOBAL_SEED)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=GLOBAL_SEED)



4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
lyamdas = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0]
results_list = []

# Пункт 4. VGD с постоянным шагом n
res_4 = run_experiment("4. VGD (постоянный шаг)", 'vanilla', False, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_4)

5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 5. VGD с переменным шагом (TimeDecayLR)
res_5 = run_experiment("5. VGD (TimeDecayLR)", 'vanilla', True, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_5)

6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 6. SGD с постоянным шагом n
res_6 = run_experiment("6. SGD (постоянный шаг)", 'sgd', False, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_6)

7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 7. SGD с переменным шагом (TimeDecayLR)
res_7 = run_experiment("7. SGD (TimeDecayLR)", 'sgd', True, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_7)

8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 8. SAG с постоянным шагом n
res_8 = run_experiment("8. SAG (постоянный шаг)", 'sag', False, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_8)

9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:

# Пункт 9. SAG с переменным шагом (TimeDecayLR)
res_9 = run_experiment("9. SAG (TimeDecayLR)", 'sag', True, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_9)

10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:

# Пункт 10. Momentum с постоянным шагом n
res_10 = run_experiment("10. Momentum (постоянный шаг)", 'momentum', False, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_10)


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 11. Momentum с переменным шагом (TimeDecayLR)
res_11 = run_experiment("11. Momentum (TimeDecayLR)", 'momentum', True, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_11)

12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 12. Adam с постоянным шагом n
res_12 = run_experiment("12. Adam (постоянный шаг)", 'adam', False, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_12)

13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
# Пункт 13. Adam с переменным шагом (TimeDecayLR)
res_13 = run_experiment("13. Adam (TimeDecayLR)", 'adam', True, X_train, y_train, X_val, y_val, X_test, y_test, lyamdas)
results_list.append(res_13)

14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [ ]:
df_summary = pd.DataFrame(results_list)

print("=========================================================================================")
print("ПУНКТ 14. СРАВНИТЕЛЬНАЯ ТАБЛИЦА МЕТОДОВ")
print("=========================================================================================")
print(df_summary.to_string(index=False))

15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [ ]:
1. лучший метод это adam, так как по таблице можно заметить, что у этого метода максимальное значение коэффициента детерминации и наименьшая квадратичная ошибка
2. тренировачный показывает, насколько хорошо машина обучилась на тренировчной выборке, а тестовая показывает насколько хорошо модель работает с новыми машинами из тестового набора
3. они помогают определить правильность обучения модели, а нужны они оба, чтобы можно было на основие их значений понять модель переобучилась или недообучилась